# Figure 4 – SOC estimation (Dataset 2)

Data are read from `data/` (written by `run_all.py`). Every figure is saved as `Figure4_<nn>_<title>.png` when it is shown.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA = Path("data")
PREFIX = "Figure4"
_plt_show = plt.show
_n = [0]


def _title(fig):
    for ax in fig.axes:
        t = ax.get_title()
        if t:
            return re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_")[:50]
    return "figure"


def _save_then_show(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        _n[0] += 1
        name = f"{PREFIX}_{_n[0]:02d}_{_title(fig)}.png"
        fig.savefig(name, dpi=fig.dpi, bbox_inches="tight")
        print("saved:", name)
    _plt_show(*args, **kwargs)


plt.show = _save_then_show

In [ ]:
d = pd.read_csv(DATA / "soc_predictions.csv")
soc_true = d["SOC_true"].values
soc_pred_oracle = d["SOC_pred"].values
df_test = d[["Mat"]]

## SOC predictions of the test samples

In [ ]:
import numpy as np
import pandas as pd

df_soc = pd.DataFrame({
    "SOC_true": soc_true,
    "SOC_pred": soc_pred_oracle,
    "Mat": df_test["Mat"].values
})

df_soc = df_soc.dropna().reset_index(drop=True)

## Predicted vs. true SOC for each material

In [ ]:
import matplotlib.pyplot as plt

for mat in df_soc["Mat"].unique():

    df_m = df_soc[df_soc["Mat"] == mat]
    if len(df_m) < 20:
        continue

    plt.figure(figsize=(3,2.5),dpi=600)

    plt.scatter(
        df_m["SOC_true"],
        df_m["SOC_pred"],
        s=14,
        alpha=0.6,
        color='#34669A'
    )

    plt.plot([0,100], [0,100], "k--", lw=1)

    plt.xlabel("True SOC (%)", fontsize=10)
    plt.ylabel("Predicted SOC (%)", fontsize=10)
    plt.title(f"SOC Prediction ({mat})", fontsize=11)

    plt.xlim(0, 100)
    plt.ylim(0, 100)

    plt.tight_layout()
    plt.show()

## SOC error distribution for each material

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_soc_error_hist_by_material(df_soc):

    for mat in df_soc["Mat"].unique():

        df_m = df_soc[df_soc["Mat"] == mat]
        err = df_m["SOC_pred"].values - df_m["SOC_true"].values
        err = err[~np.isnan(err)]

        if len(err) < 20:
            continue

        plt.figure(figsize=(1.5, 1.3),dpi=600)

        sns.histplot(
            err,
            bins=20,
            kde=True,
            color="#80A2C5"
        )

        plt.axvline(0, color="k", linestyle="--", lw=1)

        plt.xlabel("SOC Prediction Error (%)", fontsize=9)
        plt.ylabel("Count", fontsize=9)
        plt.title(f"SOC Error Distribution ({mat})", fontsize=10)

        plt.tight_layout()
        plt.show()

        df_soc = pd.DataFrame({
    "SOC_true": soc_true,
    "SOC_pred": soc_pred_oracle,
    "Mat": df_test["Mat"].values
}).dropna()

plot_soc_error_hist_by_material(df_soc)

## Predicted vs. true SOC for all test samples

In [ ]:
color='#34669A'
import matplotlib.pyplot as plt

def plot_soc_scatter(y_true, y_pred, title):
    mask = ~np.isnan(y_pred)

    plt.figure(figsize=(3, 2.5),dpi=600)
    plt.scatter(
        y_true[mask], y_pred[mask],
        s=30, alpha=0.7,color='#34669A',
    )

    lims = [0, 100]
    plt.plot(lims, lims, "k--", linewidth=1)

    plt.xlim(lims)
    plt.ylim(lims)

    plt.xlabel("True SOC (%)")
    plt.ylabel("Predicted SOC (%)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_soc_scatter(
    soc_true, soc_pred_oracle,
    "SOC Prediction"
)

## Predicted vs. true SOC for all test samples, with MAE and R²

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score

color = '#34669A'

def plot_soc_scatter(y_true, y_pred, title):
    mask = ~np.isnan(y_pred)

    y_true_valid = y_true[mask]
    y_pred_valid = y_pred[mask]

    mae = mean_absolute_error(
        y_true_valid,
        y_pred_valid
    )

    r2 = r2_score(
        y_true_valid,
        y_pred_valid
    )

    print(f"MAE = {mae:.2f}%")
    print(f"R² = {r2:.4f}")

    plt.figure(figsize=(3, 2.5), dpi=600)

    plt.scatter(
        y_true_valid,
        y_pred_valid,
        s=30,
        alpha=0.7,
        color='#34669A'
    )

    lims = [0, 100]
    plt.plot(lims, lims, "k--", linewidth=1)

    plt.xlim(lims)
    plt.ylim(lims)

    plt.xlabel("True SOC (%)")
    plt.ylabel("Predicted SOC (%)")
    plt.title(title)

    plt.tight_layout()
    plt.show()

    return mae, r2


mae, r2 = plot_soc_scatter(
    soc_true,
    soc_pred_oracle,
    "SOC Prediction"
)